### Programming for Biomedical Informatics
#### Week 3 - Data Integration & Summary Analysis

Using some of the skills we've developed working with eUtils we're now going to take two different lists of genes that use different identifiers convert them to NCBI Gene IDs and then use these to merge the data together. With the final merged data we will do some calculations and plots.

In [63]:
# Preliminaries
from Bio import Entrez
import urllib.request
import json
import xml.etree.ElementTree as ET
import pandas as pd

# load my API key from the file
with open('../api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../api_keys/ncbi_email.txt', 'r') as file:
    email = file.read().strip()

Entrez.api_key = api_key
Entrez.email = email

In [64]:
# Step 1 - Load the two lists that we cannot currenly combine

'''The first file contains a list of gene symbols

e.g.
GeneSymbol
ADAM10
ADAM17
APP
NAE1
APBB1
GAPDH
BACE1

The second file contains a list of RefSeq transcripts (mRNA), and their associated GO terms:

e.g.
RefSeqID        GOTerm  Description
NM_001320570    GO:0003824      catalytic activity
NM_001320570    GO:0016787      hydrolase activity
NM_001320570    GO:0140096      catalytic activity, acting on a protein
NM_001320570    GO:0043226      organelle
NM_001320570    GO:0005634      nucleus
NM_001320570    GO:0005794      Golgi apparatus

We are going to convert Gene Symbols and Refseq IDs to NCBI Gene IDs, and then combine the two lists into a single table.
'''

# Load the gene symbols as a pandas dataframe
gene_symbols = pd.read_csv('./GeneSymbols.tsv', sep='\t', header=0) 
# Load the RefSeq data as a pandas dataframe
refseq_data = pd.read_csv('./transcript_functions.tsv', sep='\t', header=0)


In [65]:
# view the first few rows of each dataframe
gene_symbols.head()

,GeneSymbol
0,ADAM10
1,ADAM17
2,APP
3,NAE1
4,APBB1


In [66]:
# view the first few rows of each dataframe
refseq_data.head()

,RefSeqID,GOTerm,Description
0,NM_001320570,GO:0003824,catalytic activity
1,NM_001320570,GO:0016787,hydrolase activity
2,NM_001320570,GO:0140096,"catalytic activity, acting on a protein"
3,NM_001320570,GO:0043226,organelle
4,NM_001320570,GO:0005634,nucleus


In [67]:
#Step 2 - Convert Gene Symbols to NCBI Gene IDs

# create a dictionary that maps gene symbols to gene IDs
gene_symbols_to_id = {}
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
# use eSearch to convert gene symbols to NCBI Gene IDs (for the first 10 gene symbols)
for symbol in gene_symbols['GeneSymbol'][:10]:
    params = {
        'db': 'gene',
        'term': f'{symbol}[Gene] AND human[Organism]',
        'api_key': api_key,
        'email': email,
        'usehistory': 'y'
    }
    url = base_url + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        xml = response.read()
        root = ET.fromstring(xml)
        id_list = root.find("IdList")
        gene_id = id_list.find("Id").text if id_list is not None and id_list.find("Id") is not None else None
        gene_symbols_to_id[symbol] = gene_id
        table.add_row([symbol, gene_id])
        print(f"{i+1}: {symbol} -> {gene_id}")
        
        
    
# # remembering to add API key and email
# # remembering to use the [Gene] field in the search
# # remembering to specify human
# # show the progress by printing the gene symbol and gene ID and the number of gene symbols processed so far
# # This takes about 3 minutes (NB not the quickest way!)
# '''### YOUR CODE HERE ###'''

# #user PrettyTable to display the results
# '''### YOUR CODE HERE ###'''

10: ADAM10 -> 102
10: ADAM17 -> 6868
10: APP -> 351
10: NAE1 -> 8883
10: APBB1 -> 322
10: GAPDH -> 2597
10: BACE1 -> 23621
10: BACE2 -> 25825
10: RTN3 -> 10313
10: RTN4 -> 57142


In [68]:
# Step 3 - Convert RefSeq IDs to NCBI Gene IDs

# create a dictionary that maps RefSeq IDs to Gene IDs
refseq_to_gene_id = {}
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

# use eSearch to convert RefSeq IDs to NCBI Gene IDs (for the first 10 RefSeq IDs)
for id in refseq_data['RefSeqID'].unique()[:10]:
    params = {
        'dbfrom': 'nucleotide',
        'db': 'gene',
        'api_key': api_key,
        'email': email,
    }
# remembering to add API key and email
    url = base_url + "?" + urllib.parse.urlencode(params) + f"&id={id}"
    with urllib.request.urlopen(url) as response:
        xml = response.read()
        root = ET.fromstring(xml)
        for linkset in root.findall('LinkSet'):
            gene_id = linkset.find('LinkSetDb/Link/Id').text
        refseq_to_gene_id[id] = gene_id
        print(f"{id} -> {gene_id}")
        


NM_001320570 -> 102
NM_001110 -> 102
nan -> 102
NM_003183 -> 6868
NM_001382777 -> 6868
NM_001382778 -> 6868
NM_175573 -> 11047
NM_007002 -> 11047
NM_001281438 -> 11047
NM_001281437 -> 11047


In [73]:
#user PrettyTable to display the results
import prettytable
table = prettytable.PrettyTable()
table.field_names = ["RefSeqID", "GeneID"]
for refseq_id, gene_id in list(refseq_to_gene_id.items())[:10]:
    table.add_row([refseq_id, gene_id])
print(table)

+--------------+--------+
|   RefSeqID   | GeneID |
+--------------+--------+
| NM_001320570 |  102   |
|  NM_001110   |  102   |
|     nan      |  102   |
|  NM_003183   |  6868  |
| NM_001382777 |  6868  |
| NM_001382778 |  6868  |
|  NM_175573   | 11047  |
|  NM_007002   | 11047  |
| NM_001281438 | 11047  |
| NM_001281437 | 11047  |
+--------------+--------+


In [70]:
# Step 4  - merge the refseq_to_gene_id dictionary with the refseq dataframe

# create a new column in the refseq dataframe called 'GeneID'
# fill the column with the gene IDs from the refseq_to_gene_id dictionary
refseq_data['GeneID'] = refseq_data['RefSeqID'].map(refseq_to_gene_id)

# remove rows with missing values
refseq_data = refseq_data.dropna()
# display the refseq dataframe
refseq_data.head()

,RefSeqID,GOTerm,Description,GeneID
0,NM_001320570,GO:0003824,catalytic activity,102
1,NM_001320570,GO:0016787,hydrolase activity,102
2,NM_001320570,GO:0140096,"catalytic activity, acting on a protein",102
3,NM_001320570,GO:0043226,organelle,102
4,NM_001320570,GO:0005634,nucleus,102


In [71]:
gene_symbols_to_id

{'ADAM10': '102',
 'ADAM17': '6868',
 'APP': '351',
 'NAE1': '8883',
 'APBB1': '322',
 'GAPDH': '2597',
 'BACE1': '23621',
 'BACE2': '25825',
 'RTN3': '10313',
 'RTN4': '57142'}

In [74]:
#Step 7 - combine the gene_symbol and refseq dataframes

# convert the gene_symbol_to_id dictionary to a dataframe
gene_symbols_to_id_df = pd.DataFrame(gene_symbols_to_id.items(), columns=['GeneSymbol', 'GeneID'])
# merge the refseq and gene_symbol_to_id_df dataframes on the 'GeneID' column
refseq_geneid_merge = refseq_data.merge(gene_symbols_to_id_df, on='GeneID')
# drop the GeneID column
'''### YOUR CODE HERE ###'''
refseq_geneid_merge = refseq_geneid_merge.drop(columns=['GeneID'])    
# display the combined dataframe
refseq_geneid_merge.head()

,RefSeqID,GOTerm,Description,GeneSymbol
0,NM_001320570,GO:0003824,catalytic activity,ADAM10
1,NM_001320570,GO:0016787,hydrolase activity,ADAM10
2,NM_001320570,GO:0140096,"catalytic activity, acting on a protein",ADAM10
3,NM_001320570,GO:0043226,organelle,ADAM10
4,NM_001320570,GO:0005634,nucleus,ADAM10


In [76]:
#Step 8 - Do some basic summary analysis

# display the number of rows and columns in the combined dataframe
print("shape:" ,refseq_geneid_merge.shape)
# count how many unique genes are in the combined dataframe
unique_genes = refseq_geneid_merge['GeneSymbol'].nunique()
# count how many unique GO terms are in the combined dataframe
unique_go_terms = refseq_geneid_merge['GOTerm'].nunique()
# display the number of unique genes and GO terms
print("Unique Genes:", unique_genes)
print("Unique GO terms:", unique_go_terms)
# use some plots to visualise the data (up to you!)


shape: (92, 4)
Unique Genes: 2
Unique GO terms: 30
